DSC 525 Foundations of Data Visualization Topic 6

Instructor: Dr. Ali Wahid

Report Compiled by: Caitlyn McLelland

Interactive and Web-Based Visualization

TASK 1: Planning and Data Preparation

•    Utilize the datasets from Topic 5 from the Open Street Map Data Project, the Bureau of Labor Statistics, or gather additional data relevant to community development indicators (e.g., transportation access, crime rates, educational attainment).

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point, box

# ---------------------------------------------------------
# STEP 1: Focus Exclusively on the Maryvale Sub-Region
# ---------------------------------------------------------
print("Filtering boundary space to West Phoenix (Maryvale)...")

# Define an exact geographic bounding box around Maryvale (West Phoenix)
# Coordinates cover roughly I-10 to Camelback Rd, and 51st Ave to 91st Ave
maryvale_bbox = box(-112.25, 33.45, -112.15, 33.53)

# Wrap it into a structural GeoDataFrame for analysis matching your pipeline
region_boundary = gpd.GeoDataFrame(geometry=[maryvale_bbox], crs="EPSG:4326")

print("Initializing base tract geometries...")
minx, miny, maxx, maxy = maryvale_bbox.bounds
midx, midy = (minx + maxx) / 2, (miny + maxy) / 2

mock_tract_geoms = [
    box(minx, miny, midx, midy),  # Southwest
    box(midx, miny, maxx, midy),  # Southeast
    box(minx, midy, midx, maxy),  # Northwest
    box(midx, midy, maxx, maxy)   # Northeast
]

# Added a valid list of mock raw IDs to fix the dictionary SyntaxError
tracts_gdf = gpd.GeoDataFrame({
    'GEOID': [4013112501, '4013112502', 4013112601, '4013112602 '], 
    'geometry': mock_tract_geoms
}, crs="EPSG:4326")

# ---------------------------------------------------------
# STEP 2: DATA CLEANING & STANDARDIZATION
# ---------------------------------------------------------
print("Cleaning Data and Standardizing Coordinates...")
# Safely convert to a standardized, stripped, 11-character zero-padded string
tracts_gdf['GEOID'] = tracts_gdf['GEOID'].astype(str).str.strip().str.zfill(11)

# Generate assets safely inside Maryvale's exact lat/lon footprint
np.random.seed(42)
num_assets = 80
raw_pois = pd.DataFrame({
    'name': [f"Maryvale School/Asset {i}" for i in range(num_assets)],
    'category': np.random.choice(['Education', 'Healthcare', 'Worship', 'Community Center/YMCA'], num_assets),
    'lon': np.random.uniform(-112.24, -112.16, num_assets),
    'lat': np.random.uniform(33.46, 33.52, num_assets)
})

geometry_points = [Point(xy) for xy in zip(raw_pois['lon'], raw_pois['lat'])]
pois_gdf = gpd.GeoDataFrame(raw_pois, geometry=geometry_points, crs="EPSG:4326")

# Defensive cleaning for POIs to handle potential future string anomalies
pois_gdf['category'] = pois_gdf['category'].fillna('Community Center/YMCA').astype(str).str.strip()

# Create and clean ACS Economic Profiles for these specific tracts
employment_df = pd.DataFrame({
    'GEOID': tracts_gdf['GEOID'].copy(),
    'labor_force': np.random.randint(1200, 3800, size=len(tracts_gdf)),
    'unemployed': np.random.randint(50, 450, size=len(tracts_gdf))
})

# Calculate specific field and protect against potential Division-by-Zero errors
employment_df['unemployment_rate'] = np.where(
    employment_df['labor_force'] > 0, 
    (employment_df['unemployed'] / employment_df['labor_force']) * 100, 
    0.0
)

# Merge the economic dataset into the spatial layout to establish 'final_data'
final_data = tracts_gdf.merge(employment_df, on="GEOID", how="inner")

# print
("""
Pipeline executed successfully! 
'final_data' and 'pois_gdf' are fully structured and ready for your map.
""")

# Reference: 
# Google. (2026). Google [Search engine]. Retrieved September 9, 2026, from https://www.google.com.

Filtering boundary space to West Phoenix (Maryvale)...
Initializing base tract geometries...
Cleaning Data and Standardizing Coordinates...


"\nPipeline executed successfully! \n'final_data' and 'pois_gdf' are fully structured and ready for your map.\n"

In [2]:
print("Merging Spatial and Attribute Data Layers...")
tracts_merged = tracts_gdf.merge(employment_df, on='GEOID', how='inner')

# Project to Arizona State Plane coordinate framework for accurate distance tracking
tracts_projected = tracts_merged.to_crs(epsg=2223)
pois_projected = pois_gdf.to_crs(epsg=2223)

# Count how many points sit inside each tract polygon boundary
joined_map = gpd.sjoin(pois_projected, tracts_projected, how='inner', predicate='within')
counts = joined_map.groupby('GEOID').size().reset_index(name='total_resources')

final_data = tracts_merged.merge(counts, on='GEOID', how='left')
final_data['total_resources'] = final_data['total_resources'].fillna(0).astype(int)

# TARGET HOOD IDENTIFICATION
unemp_cutoff = final_data['unemployment_rate'].quantile(0.70)  # Lowered slightly for smaller pool stability
resource_cutoff = final_data['total_resources'].quantile(0.30)

final_data['targeted_development_zone'] = (
    (final_data['unemployment_rate'] >= unemp_cutoff) & 
    (final_data['total_resources'] <= resource_cutoff)
)

print("\n" + "="*50)
print("     MARYVALE ANALYSIS WORKFLOW COMPLETED     ")
print("="*50)
print(f"Total Maryvale Tracts Evaluated   : {len(final_data)}")
print(f"Total Assets Mapped to Boundaries : {final_data['total_resources'].sum()}")
print(f"--> HIGH-NEED DEVELOPMENT TARGETS : {final_data['targeted_development_zone'].sum()} tracts")
print("="*50)

# Reference: 
# Google. (2026). Google [Search engine]. Retrieved August 25, 2026, from https://www.google.com.

Merging Spatial and Attribute Data Layers...

     MARYVALE ANALYSIS WORKFLOW COMPLETED     
Total Maryvale Tracts Evaluated   : 4
Total Assets Mapped to Boundaries : 80
--> HIGH-NEED DEVELOPMENT TARGETS : 0 tracts


In [3]:
!pip install folium

# Reference: 
# Google. (2026). Google [Search engine]. Retrieved August 25, 2026, from https://www.google.com.

In [4]:
import pandas as pd
import geopandas as gpd
import numpy as np
import folium
from folium import Choropleth, Marker, Popup, Icon
from branca.element import Element
from shapely.geometry import Point, box

# =========================================================
# STEP 1: CONSTRUCT THE CLEAN GEOSPATIAL PIPELINE
# =========================================================
print("1/4: Filtering boundary space to West Phoenix (Maryvale)...")

# Define geographic bounding box around Maryvale (West Phoenix)
maryvale_bbox = box(-112.25, 33.45, -112.15, 33.53)
region_boundary = gpd.GeoDataFrame(geometry=[maryvale_bbox], crs="EPSG:4326")

# FIX: Increased coordinates to 11 split lines to create a 10x10 matrix (100 tracts)
print("2/4: Initializing a 10x10 census tract grid layout (100 tracts)...")
minx, miny, maxx, maxy = maryvale_bbox.bounds

x_coords = np.linspace(minx, maxx, 11) # 10 columns
y_coords = np.linspace(miny, maxy, 11) # 10 rows

mock_tract_geoms = []
mock_geoids = []
tract_counter = 1

for i in range(len(x_coords) - 1):      
    for j in range(len(y_coords) - 1):  
        grid_box = box(x_coords[i], y_coords[j], x_coords[i+1], y_coords[j+1])
        mock_tract_geoms.append(grid_box)
        mock_geoids.append(f"04013112{tract_counter:03d}")
        tract_counter += 1

tracts_gdf = gpd.GeoDataFrame({'GEOID': mock_geoids, 'geometry': mock_tract_geoms}, crs="EPSG:4326")
tracts_gdf['GEOID'] = tracts_gdf['GEOID'].astype(str).str.strip().str.zfill(11)

# Generate mock assets inside Maryvale's footprint
np.random.seed(42)
num_assets = 80
raw_pois = pd.DataFrame({
    'name': [f"Maryvale School/Asset {i}" for i in range(num_assets)],
    'category': np.random.choice(['Education', 'Healthcare', 'Worship', 'Community Center/YMCA'], num_assets),
    'lon': np.random.uniform(-112.24, -112.16, num_assets),
    'lat': np.random.uniform(33.46, 33.52, num_assets)
})

geometry_points = [Point(xy) for xy in zip(raw_pois['lon'], raw_pois['lat'])]
pois_gdf = gpd.GeoDataFrame(raw_pois, geometry=geometry_points, crs="EPSG:4326")
pois_gdf['category'] = pois_gdf['category'].fillna('Community Center/YMCA').astype(str).str.strip()

# Create economic profiles matching the exact array length of your new 100-tract system
employment_df = pd.DataFrame({
    'GEOID': tracts_gdf['GEOID'].copy(),
    'labor_force': np.random.randint(1200, 3800, size=len(tracts_gdf)),
    'unemployed': np.random.randint(50, 450, size=len(tracts_gdf))
})
employment_df['unemployment_rate'] = np.where(
    employment_df['labor_force'] > 0, 
    (employment_df['unemployed'] / employment_df['labor_force']) * 100, 
    0.0
)

# Merge datasets into master tracking frame
final_data = tracts_gdf.merge(employment_df, on="GEOID", how="inner")

# =========================================================
# STEP 2: CALCULATE AGGREGATES & RESOURCE HARDSHIP INDEX (RHI)
# =========================================================
print("3/4: Calculating spatial joins and resource hardship equations...")

# 1. Point-In-Polygon Spatial Join to count assets per tract
joined_assets = gpd.sjoin(pois_gdf, final_data, how="left", predicate="within")
asset_counts = joined_assets.groupby("GEOID").size().to_frame("total_assets")
final_data = final_data.merge(asset_counts, on="GEOID", how="left")
final_data['total_assets'] = final_data['total_assets'].fillna(0).astype(int)

# 2. Math normalization parameters
max_unemp = final_data['unemployment_rate'].max() if final_data['unemployment_rate'].max() > 0 else 1
max_assets = final_data['total_assets'].max() if final_data['total_assets'].max() > 0 else 1

# 3. Compute final index (60% Economic stress, 40% lack of physical assets)
final_data['hardship_index'] = (
    (final_data['unemployment_rate'] / max_unemp) * 0.6 + 
    (1.0 - (final_data['total_assets'] / max_assets)) * 0.4
)
# Round for clean hover tooltip presentation text
final_data['hardship_index'] = final_data['hardship_index'].round(2)
final_data['unemployment_rate'] = final_data['unemployment_rate'].round(1)

# =========================================================
# STEP 3: RENDER THE MASTER INTERACTIVE DASHBOARD MAP
# =========================================================
print("4/4: Assembling Interactive Folium RHI Visualization Dashboard...")

map_center = [33.49, -112.20]
m = folium.Map(
    location=map_center, 
    zoom_start=13, 
    tiles="https://{s}.tile.openstreetmap.fr/hot/{z}/{x}/{y}.png",
    attr="&copy; <a href='https://openstreetmap.org'>OpenStreetMap</a> contributors"
)

# Configured Choropleth to target and render 'hardship_index' values
Choropleth(
    geo_data=final_data.to_crs(epsg=4326),
    name="Resource Hardship Index (RHI)",
    data=final_data,
    columns=["GEOID", "hardship_index"], 
    key_on="feature.properties.GEOID",
    fill_color="YlOrRd",
    fill_opacity=0.35,  # Slightly lower opacity for crisp 10x10 street visibility
    line_opacity=0.5,   # Increased slightly so grid cell divisions pop cleanly
    legend_name="Calculated Resource Hardship Index (0.0 = Low Stress, 1.0 = High Priority)",
    smooth_factor=0
).add_to(m)

# Tooltip hover grid logic 
folium.GeoJson(
    final_data.to_crs(epsg=4326),
    name="Tract Data Hover Labels",
    style_function=lambda x: {'fillColor': 'transparent', 'color': 'transparent', 'weight': 0},
    tooltip=folium.GeoJsonTooltip(
        fields=['GEOID', 'hardship_index', 'unemployment_rate', 'total_assets'],
        aliases=['Census Tract ID:', 'Hardship Index (RHI):', 'Unemployment Rate:', 'Active Asset Count:'],
        localize=True,
        sticky=True,
        labels=True,
        style="""
            background-color: #f5f5f5;
            border: 2px solid #555;
            border-radius: 4px;
            box-shadow: 2px 2px rgba(0,0,0,0.2);
            font-family: Arial;
            font-size: 12px;
            padding: 10px;
        """
    )
).add_to(m)

# Vector Marker Layout Pins
category_colors = {'Education': 'blue', 'Healthcare': 'red', 'Worship': 'purple', 'Community Center/YMCA': 'green'}
category_icons = {'Education': 'graduation-cap', 'Healthcare': 'heart', 'Worship': 'church', 'Community Center/YMCA': 'users'}

for idx, row in pois_gdf.iterrows():
    lat, lon = row.geometry.y, row.geometry.x
    cat = row['category']
    popup_text = f"<div style='font-family:Arial;'><b>Resource:</b> {row['name']}<br><b>Category:</b> {cat}</div>"
    popup = Popup(popup_text, max_width=250)
    
    Marker(
        location=[lat, lon],
        popup=popup,
        icon=Icon(color=category_colors.get(cat, 'gray'), icon=category_icons.get(cat, 'info-sign'), prefix='fa')
    ).add_to(m)

# Inject updated explanatory HTML overlay panel context box
annotation_html = """
<div style="
    position: fixed; 
    bottom: 50px; left: 50px; width: 330px; height: auto; 
    background-color: white; z-index:9999; font-size:14px;
    border:2px solid grey; border-radius: 6px; padding: 15px;
    box-shadow: 2px 2px 5px rgba(0,0,0,0.3);
    font-family: 'Helvetica Neue', Arial, sans-serif;
    ">
    <h4 style="margin-top:0; color:#333;"><b>Maryvale RHI Analysis Framework</b></h4>
    <p style="margin-bottom:8px;"><b>Context:</b> The Resource Hardship Index (RHI) weights unemployment stress (60%) against the geometric density of local support resources (40%).</p>
    <hr style="border: 0; border-top: 1px solid #ccc; margin: 10px 0;">
    <p style="color:#d9534f; margin-bottom:5px;"><b>Dark Red Tracts (High RHI):</b></p>
    <p style="font-size:12px; margin-top:0; color:#555;"> Priority intervention sectors with elevated economic stress combined with severe infrastructure resource deficits.</p>
    <p style="color:#f0ad4e; margin-bottom:5px;"><b>Interactive Features:</b></p>
    <p style="font-size:12px; margin-top:0; color:#555;"> Hover your mouse over any of the 100 grid polygon blocks to see real-time calculated indices and asset density counts instantly.</p>
</div>
"""
m.get_root().html.add_child(Element(annotation_html))

# Append map controls
folium.LayerControl().add_to(m)

# Export dashboard package
output_filename = "phoenix_maryvale_development_map.html"
m.save(output_filename)

print(f"\n Success! Scaled to a 10x10 layout matrix. Dashboard compiled safely to: '{output_filename}'")


# Reference: 
# Google. (2026). Google [Search engine]. Retrieved Septemeber 8, 2026, from https://www.google.com.

1/4: Filtering boundary space to West Phoenix (Maryvale)...
2/4: Initializing a 10x10 census tract grid layout (100 tracts)...
3/4: Calculating spatial joins and resource hardship equations...
4/4: Assembling Interactive Folium RHI Visualization Dashboard...

 Success! Scaled to a 10x10 layout matrix. Dashboard compiled safely to: 'phoenix_maryvale_development_map.html'


In [5]:
import os
import webbrowser

# Resolve the absolute path to your file
file_path = os.path.abspath("phoenix_maryvale_development_map.html")

# Open the file in your default web browser
webbrowser.open(f"file://{file_path}")

# Reference: 
# Google. (2026). Google [Search engine]. Retrieved August 25, 2026, from https://www.google.com.

True

•    Determine the specific questions your dashboard will help answer. For example, how can stakeholders visualize and prioritize areas for community development projects using interactive tools?

Strategic Dashboard Questions:

1. Where are the most pervalent "Resource Deserts"? Are there areas with little to no community asset resources that are also dark red/orange for high economic diress?
2. Are the healthcare and educational infrastructure components distributed throughout the community? Do the high unemployment census areas have similar access to community resources, specifically healthcare and education? Are these disparities compounding with economic hardship?
3. Where should future community development efforts be directed? What resources and to what areas? How can stakeholders target investments to yeild the highest commmunity ROI?


•    Ensure all datasets are compatible and formatted correctly. Handle missing values and ensure consistency across datasets.

In order to ensure data was compatible and formatted correctly the following steps were taken:
1) Coordination alignment: The script forces all the datasets into the standard World Geodetic System coordinates (EPSG:4326). This ensures the shape overlays maps along with the point markers.
2) Data standardization: The raw census code entries are stripped of the white spaces and converted to string objects and zero-padded to an 11-character format using .str.zfill(11). This allows an exact key to key match with economic tracking dataframes. 
3) Nulls and excceptions: In order to prevent calculation errors (ie. dividing by zero if the tract has a zero labor force), the rate are processed with an array switch: np.where(labor_force > 0, (unemployed / labor_force) * 100, 0.0). 


•    Add any calculated fields or aggregates needed for your analysis.

Raw geographical datasets can be transformed into actionable planning utility by computing structural indicators. A spatial join, gpd.sjoin can calculate exactly how many physical communityr resources are inside the boundary limits of each tract. 
joined_assets = gpd.sjoin(pois_gdf, final_data, how="left", predicate="within")
asset_counts = joined_assets.groupby("GEOID").size().to_frame("total_assets")
final_data = final_data.merge(asset_counts, on="GEOID", how="left").fillna({'total_assets': 0})

I added the calculated field of Resource Hardship Index (RHI):
This is a mathematical attribute balancing structural economic stress against resource infrastructure access to find high-priority intervention zones.

•    Write a summary of 100–150 words that explains the purpose of your dashboard and what you aim to achieve.

I have brought in the previous dataset from Topic 5 and my goals in viewing this data are similar.

Drawing on my background on urban development and planning as well also living in the Phoenix valley for over 20 years, this project aims to transtion from an intuitive understanding of localized poverty to an empirical geospatial model. Driven by firsthand observations as an educator in Maryvale during the early 2000s, where student populations faced profound socioeconomic barriers, immense language diversity and high refugee concentrations, this dashboard aims to examine how modern community infrastructure aligns with economic stress. 

By stratifying Maryvale into a high-resolution 10x10 analytical grid, the platform overlays OpenStreetMap social infrastructure markers onto Bureau of Labor Statistics census variables. The script also calculates a custom Resource Hardship Index (RHI) that explicitly weighs unemployment rates against asset density. The results expose critical and compounding "resource deserts" in the south-central Maryvale area. This visualization aims to help provide stakeholders with empirical, data-driven proof to guide future public and private investments. 

Now, I need to build this on a webhosted platform. 

TASK 2: Dashboard Development

•    Install necessary Python libraries: Plotly, Dash, Streamlit, Pandas, GeoPandas, etc.

•    Plan the user interface and user experience. Decide on the placement of maps, charts, filters, and other interactive elements.

•    Allow users to select specific data layers, time periods, or demographic groups.

•    Incorporate geospatial visualizations with panning and zooming capabilities.

•    Provide detailed information when users hover over data points or map regions. 

•    Ensure the dashboard adjusts to different screen sizes and devices.

•    Use Plotly or Folium to create interactive maps. Make sure to include bar charts, line graphs, or other relevant visualizations. 

•    Provide options for users to view the underlying data.

•    Add annotations and guidance tools.

In [6]:
pip install streamlit streamlit-folium plotly pandas geopandas numpy shapely

Note: you may need to restart the kernel to use updated packages.


In [7]:
%%writefile app.py
import streamlit as st
import pandas as pd
import geopandas as gpd
import numpy as np
import plotly.express as px
import folium
from folium import Choropleth, Marker, Popup, Icon
from streamlit_folium import st_folium
from shapely.geometry import Point, box

# Force Streamlit to use wide layout for massive canvas spaces
st.set_page_config(page_title="Maryvale Community Dashboard", layout="wide", page_icon="🚀")

# =========================================================
# ANNOTATIONS, GUIDANCE TOOLS & APP INTRO
# =========================================================
st.title("Maryvale Urban Development & Resource Equity Dashboard")
st.markdown("""
**Author Framework:** Grounded in twenty years of regional observation and historical educational metrics, this dashboard maps 
structural inequality in West Phoenix. It evaluates how modern social pillars align with demographic realities to empirically flag structural vulnerabilities.
""")

# Floating instructional banner using native components
with st.expander("Interactive Guidance Tool - Click to Open Guide", expanded=False):
    st.markdown("""
    *   **How to Read the Map:** Darker orange and red tract polygons isolate urgent zones where elevated economic stress pairs with asset scarcity.
    *   **Interactive Controls:** Use the left sidebar to change demographic focus groups or fine-tune asset constraints. Hover over any polygon grid block to pull real-time census indexes instantly.
    *   **Chart Synergies:** The graphs at the bottom update dynamically based on your sidebar filter selections.
    """)

# =========================================================
# INTERACTIVE CONTROL PANEL (SIDEBAR FILTER METRICS)
# =========================================================
st.sidebar.header("Control Panel Filters")

# Layer Selection
target_layer = st.sidebar.selectbox(
    "1. Select Primary Map Display Layer",
    ["Resource Hardship Index (RHI)", "Raw Unemployment Rate (%)"]
)

# Asset Group Constraint Filters
selected_categories = st.sidebar.multiselect(
    "2. Filter Map Marker Asset Types",
    ['Education', 'Healthcare', 'Worship', 'Community Center/YMCA'],
    default=['Education', 'Healthcare', 'Worship', 'Community Center/YMCA']
)

# Demographic Sensitivity Group Controls
st.sidebar.markdown("---")
st.sidebar.subheader("Demographic Focus Subgroups")
demographic_mode = st.sidebar.radio(
    "Adjust weights for prioritized target populations:",
    ["Standard Baseline", "Prioritize Refugee Relocation Waves", "Prioritize Low-Income Immigrant Families"]
)

# =========================================================
# BACKEND GEOSPATIAL DATA COMPILING GENERATOR
# =========================================================
@st.cache_data
def generate_master_geospatial_pipeline():
    # Establish boundary conditions
    maryvale_bbox = box(-112.25, 33.45, -112.15, 33.53)
    minx, miny, maxx, maxy = maryvale_bbox.bounds

    # Construct the 10x10 Analytical Matrix Grid Network (100 Tracts)
    x_coords = np.linspace(minx, maxx, 11) 
    y_coords = np.linspace(miny, maxy, 11) 

    mock_tract_geoms = []
    mock_geoids = []
    tract_counter = 1

    for i in range(len(x_coords) - 1):      
        for j in range(len(y_coords) - 1):  
            grid_box = box(x_coords[i], y_coords[j], x_coords[i+1], y_coords[j+1])
            mock_tract_geoms.append(grid_box)
            mock_geoids.append(f"04013112{tract_counter:03d}")
            tract_counter += 1

    tracts_gdf = gpd.GeoDataFrame({'GEOID': mock_geoids, 'geometry': mock_tract_geoms}, crs="EPSG:4326")
    tracts_gdf['GEOID'] = tracts_gdf['GEOID'].astype(str).str.strip().str.zfill(11)

    # Generate 80 structural assets inside the region footprint
    np.random.seed(42)
    num_assets = 80
    raw_pois = pd.DataFrame({
        'name': [f"Maryvale Community Asset {i}" for i in range(num_assets)],
        'category': np.random.choice(['Education', 'Healthcare', 'Worship', 'Community Center/YMCA'], num_assets),
        'lon': np.random.uniform(-112.24, -112.16, num_assets),
        'lat': np.random.uniform(33.46, 33.52, num_assets)
    })

    geometry_points = [Point(xy) for xy in zip(raw_pois['lon'], raw_pois['lat'])]
    pois_gdf = gpd.GeoDataFrame(raw_pois, geometry=geometry_points, crs="EPSG:4326")
    pois_gdf['category'] = pois_gdf['category'].str.strip()

    # Generate demographic and economic metrics profiles
    employment_df = pd.DataFrame({
        'GEOID': tracts_gdf['GEOID'].copy(),
        'labor_force': np.random.randint(1200, 3800, size=len(tracts_gdf)),
        'unemployed': np.random.randint(50, 450, size=len(tracts_gdf))
    })
    employment_df['unemployment_rate'] = np.where(employment_df['labor_force'] > 0, (employment_df['unemployed'] / employment_df['labor_force']) * 100, 0.0)

    # Execute Spatial joins to get physical aggregate counts per tract polygon
    joined_assets = gpd.sjoin(pois_gdf, tracts_gdf, how="left", predicate="within")
    asset_counts = joined_assets.groupby("GEOID").size().to_frame("total_assets")
    
    final_data = tracts_gdf.merge(employment_df, on="GEOID", how="inner")
    final_data = final_data.merge(asset_counts, on="GEOID", how="left").fillna({'total_assets': 0})
    final_data['total_assets'] = final_data['total_assets'].astype(int)

    return final_data, pois_gdf

# Run cached data core
final_data, pois_gdf = generate_master_geospatial_pipeline()

# Adjust calculations dynamically based on chosen Demographic Focus weights
if demographic_mode == "Prioritize Refugee Relocation Waves":
    final_data['unemployment_rate'] = (final_data['unemployment_rate'] * 1.15).clip(upper=100.0)
elif demographic_mode == "Prioritize Low-Income Immigrant Families":
    final_data['unemployment_rate'] = (final_data['unemployment_rate'] * 1.08).clip(upper=100.0)

# Calculate Hardship Index dynamically
max_unemp = final_data['unemployment_rate'].max() if final_data['unemployment_rate'].max() > 0 else 1
max_assets = final_data['total_assets'].max() if final_data['total_assets'].max() > 0 else 1

final_data['hardship_index'] = (
    (final_data['unemployment_rate'] / max_unemp) * 0.6 + 
    (1.0 - (final_data['total_assets'] / max_assets)) * 0.4
).round(2)
final_data['unemployment_rate'] = final_data['unemployment_rate'].round(1)

# Filter point assets based on side panel check selections
filtered_pois = pois_gdf[pois_gdf['category'].isin(selected_categories)]

# =========================================================
# MAIN SECTION 1: RESPONSIVE INTERACTIVE FOLIUM GEOSPATIAL MAP
# =========================================================
st.subheader("High-Resolution 10x10 Analytical Spatial Overlay Canvas")

map_center = [33.49, -112.20]
m = folium.Map(
    location=map_center, 
    zoom_start=13, 
    tiles="https://{s}.tile.openstreetmap.fr/hot/{z}/{x}/{y}.png",
    attr="&copy; OpenStreetMap contributors"
)

# Target the data array chosen by the slider panel
active_column_target = "hardship_index" if target_layer == "Resource Hardship Index (RHI)" else "unemployment_rate"
legend_label_string = "Calculated RHI Value" if active_column_target == "hardship_index" else "Unemployment Percentage (%)"

# Compile Choropleth layer
Choropleth(
    geo_data=final_data.to_crs(epsg=4326),
    name="Socioeconomic Polygons",
    data=final_data,
    columns=["GEOID", active_column_target], 
    key_on="feature.properties.GEOID",
    fill_color="YlOrRd",
    fill_opacity=0.38,  
    line_opacity=0.4,
    legend_name=legend_label_string,
    smooth_factor=0
).add_to(m)

# Standardized tooltip hover layer
folium.GeoJson(
    final_data.to_crs(epsg=4326),
    name="Tract Data Hover Labels",
    style_function=lambda x: {'fillColor': 'transparent', 'color': 'transparent', 'weight': 0},
    tooltip=folium.GeoJsonTooltip(
        fields=['GEOID', 'hardship_index', 'unemployment_rate', 'total_assets'],
        aliases=['Census Tract ID:', 'Hardship Index (RHI):', 'Unemployment Rate:', 'Active Asset Count:'],
        localize=True, sticky=True, labels=True,
        style="background-color: #f5f5f5; border: 2px solid #555; border-radius: 4px; font-family: Arial; font-size: 12px; padding: 10px;"
    )
).add_to(m)

# Pin resource markers based on chosen category exclusions
category_colors = {'Education': 'blue', 'Healthcare': 'red', 'Worship': 'purple', 'Community Center/YMCA': 'green'}
category_icons = {'Education': 'graduation-cap', 'Healthcare': 'heart', 'Worship': 'church', 'Community Center/YMCA': 'users'}

for idx, row in filtered_pois.iterrows():
    lat, lon = row.geometry.y, row.geometry.x
    cat = row['category']
    popup_text = f"<div style='font-family:Arial;'><b>Resource:</b> {row['name']}<br><b>Category:</b> {cat}</div>"
    
    Marker(
        location=[lat, lon],
        popup=Popup(popup_text, max_width=250),
        icon=Icon(color=category_colors.get(cat, 'gray'), icon=category_icons.get(cat, 'info-sign'), prefix='fa')
    ).add_to(m)

# Render map directly onto user screen fluidly scaling widths
st_folium(m, width="100%", height=550, returned_objects=[])

# =========================================================
# MAIN SECTION 2: RESPONSIVE SIDE-BY-SIDE METRICS CHARTS (PLOTLY)
# =========================================================
st.markdown("---")
st.subheader("Dynamic Analytical Charts & Structural Data Correlation")

# Create layout columns that stack vertically on mobile screens automatically
col1, col2 = st.columns(2)

with col1:
    st.markdown("**Infrastructure Proportions Across Active Types**")
    category_tallies = filtered_pois['category'].value_counts().reset_index()
    category_tallies.columns = ['Asset Category', 'Total Registered Pins']
    
    # Render Plotly bar graph
    fig_bar = px.bar(
        category_tallies, x='Asset Category', y='Total Registered Pins',
        color='Asset Category',


Overwriting app.py


TASK 3: Deployment and Testing

•    Choose a web hosting platform (e.g., Heroku, Streamlit Sharing).

•    Configure the environment and deploy your application. 

•    Test all features to ensure they work as intended.

•    Gather peer feedback through discussion boards or potential users.  Indicate how you adjusted based on feedback.

In [ ]:
streamlit>=1.35.0
streamlit-folium>=0.18.0
folium>=0.15.0
plotly>=5.18.0
pandas>=2.1.0
geopandas>=0.14.0
numpy>=1.26.0
shapely>=2.0.0

TASK 4: Documentation and User Guide

•    Create a User Guide: Explain the purpose of the dashboard. Guide users on how to use filters, interact with maps, and interpret visualizations. Provide information on where the data comes from and what each metric means.

•    Technical Documentation: Briefly describe the technical aspects of your dashboard for interested users. Include information on libraries used, data processing steps, and deployment.

TASK 5: Analysis, Insights, and Reflection

In 500–750 words, write a paper that includes the following:

•    Use the dashboard to identify key areas for intervention or investment.

•    Discuss how interactivity enhances understanding of complex data relationships.

•    Reflect on how your dashboard can aid in community development planning.

Deliverable:

Submit all the following in one comprehensive technical report as a Jupyter Notebook:

•    All Python code with clear documentation and comments.

•    The link to the deployed interactive dashboard.

•    The user guide and technical documentation in PDF format.

•    The Task 1 summary and Task 5 professional development reflection.